In [1]:
import pandas as pd

df = pd.read_csv("hf://datasets/Aiman1234/Interview-questions/information.csv")

/Users/nehadevarapalli/workspace/InterviewGraph/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
columns_with_null = df.isnull().any()
rows_with_null = df.isnull().any(axis=1)
print(columns_with_null)
print(rows_with_null)

Number        False
language      False
 level        False
Questions     False
Answers        True
Unnamed: 5     True
dtype: bool
0      True
1      True
2      True
3      True
4      True
       ... 
491    True
492    True
493    True
494    True
495    True
Length: 496, dtype: bool


In [3]:
df = df.drop(columns=['Unnamed: 5'])
df.head()

,Number,language,level,Questions,Answers
0,1,Java,Easy,What is Java?,"Java is a high-level, class-based, object-orie..."
1,2,Java,Easy,What is the difference between JDK and JRE?,JDK (Java Development Kit) is for development ...
2,3,Java,Easy,Explain the main features of Java.,Some main features of Java include platform in...
3,4,Java,Easy,What is the difference between == and equals()...,"== is used to compare references, while equals..."
4,5,Java,Easy,What is the purpose of the 'static' keyword in...,The 'static' keyword is used to create class-l...


In [4]:
df = df.dropna()
df.head()

,Number,language,level,Questions,Answers
0,1,Java,Easy,What is Java?,"Java is a high-level, class-based, object-orie..."
1,2,Java,Easy,What is the difference between JDK and JRE?,JDK (Java Development Kit) is for development ...
2,3,Java,Easy,Explain the main features of Java.,Some main features of Java include platform in...
3,4,Java,Easy,What is the difference between == and equals()...,"== is used to compare references, while equals..."
4,5,Java,Easy,What is the purpose of the 'static' keyword in...,The 'static' keyword is used to create class-l...


## Load the data into Snowflake

In [5]:
import snowflake.connector
from dotenv import load_dotenv
import os

# Load environment variables from .env file
load_dotenv()

try:
    # Establish connection to Snowflake
    conn = snowflake.connector.connect(
        account=os.getenv("SNOWFLAKE_ACCOUNT"),
        user=os.getenv("SNOWFLAKE_USER"),
        password=os.getenv("SNOWFLAKE_PASSWORD"),
        warehouse=os.getenv("SNOWFLAKE_WAREHOUSE"),
        database=os.getenv("SNOWFLAKE_DATABASE"),
        schema=os.getenv("SNOWFLAKE_SCHEMA")
    )

    cursor = conn.cursor()
    print("Connection established successfully.")

except snowflake.connector.errors.Error as e:
    print(f"Error connecting to Snowflake: {e}")

Connection established successfully.


In [6]:
# resolving the non-standar index error encountered during write_pandas
df = df.reset_index(drop=True)

print(df.index)

RangeIndex(start=0, stop=490, step=1)


In [7]:
from snowflake.connector.pandas_tools import write_pandas

table_name = "interview_questions"
success, nchunks, nrows, _ = write_pandas(conn, df, table_name, auto_create_table=True)
print(f"Uploaded {nrows} rows in {nchunks} chunks")

cursor.close()
conn.close()

Uploaded 490 rows in 1 chunks
